# nanochat · 可复现分析 notebook

> 从 0 到 99 分钟跑通 GPT-2 · 6 个月 380 commits 的速度战争

**完整报告**：[`nanochat.html`](nanochat.html)（45K / ~13.6K 字）

```bash
git clone https://github.com/karpathy/nanochat ~/auto-research/nanochat
```

## 执行摘要
- 380 commits / 6 个月（2025-10-13 → 2026-04-14）/ 70+ 贡献者
- **leaderboard 加速**：OpenAI 原版 GPT-2 168h → nanochat 1.65h（**100× 加速**）
- 核心哲学：单节点 + `--depth` 单旋钮 + CORE 指标
- 杠杆顺序：**数据换血 > fp8 > 代理注入 > 批大小自动化**
- 最稀缺资产：`dev/LOG.md`（公开 lab notebook，含负面结果）

In [ ]:
import subprocess
from pathlib import Path
from collections import Counter
REPO = Path.home() / 'auto-research' / 'nanochat'
def git(*args): return subprocess.check_output(['git', '-C', str(REPO), *args], text=True)
print('commits:', git('rev-list', '--count', 'HEAD').strip())
print('first:', git('log', '--reverse', '--pretty=format:%h %ad %s', '--date=short').splitlines()[0][:80])

## 1. 按月 commit 分布 — 双峰节奏

In [ ]:
months = git('log', '--pretty=format:%ad', '--date=format:%Y-%m').strip().splitlines()
for m in sorted(set(months)):
    c = months.count(m)
    bar = '█' * min(c, 100)
    print(f'  {m}  {c:>3}  {bar}')

**双峰**：2025-10（发布冲刺 101 条）+ 2026-01（科研觉醒 97 条）。2026-04 只有 7 条——进入维护期。

## 2. Leaderboard 加速拆解

In [ ]:
# leaderboard 关键词搜索
for kw in ['fp8', 'ClimbMix', 'autoresearch', 'batch size']:
    hits = git('log', '-i', f'--grep={kw}', '--pretty=format:%h %ad %s', '--date=short').strip()
    print(f'\n### {kw}')
    for l in hits.splitlines()[:5]:
        print(' ', l)

### 加速拆解表

| 时间 | 技术 | 单次加速 | commit |
|---|---|---|---|
| Jan 29 | d24 baseline | **3.04h** | `348fbb3` |
| Feb 2 | d26 + fp8 | -4.3% → 2.91h | `a67eba3` |
| Feb 5 | batch 1M tokens | -5.2% → 2.76h | `2c062aa` |
| Mar 4 | **ClimbMix 换数据** | **-26.8%** → 2.02h | `324e69c` |
| Mar 9 | autoresearch round 1 | -10.9% → 1.80h | `6ed7d1d` |
| Mar 14 | autoresearch round 2 | -8.3% → 1.65h | `a825e63` |

**反直觉观察**：一次数据换血（27%）> 其他所有架构/精度/批大小优化累加。**数据 > 算法**。

## 3. fp8 的 3 周挣扎 — commit msg 里的真实情绪

In [ ]:
# 从失败到成功的完整轨迹
print('--- 失败那一天 ---')
print(git('show', '-s', '--format=%h %ad%n%s%n%n%b', '238353c', '--date=short'))
print('\n--- 3 周后的胜利 ---')
print(git('show', '-s', '--format=%h %ad%n%s', 'a67eba3', '--date=short'))

**学到的**：真实 commit msg 可以写情绪（"i suffered"），只要足够具体。这让 git 历史可读、可挖掘。

## 4. `dev/LOG.md` — 最稀缺的副产品

In [ ]:
log_md = REPO / 'dev' / 'LOG.md'
if log_md.exists():
    content = log_md.read_text()
    print(f'dev/LOG.md: {len(content.splitlines())} 行')
    
    # 数负面结果标记
    neg_markers = ['did not help', 'no benefit', 'Negative', 'not worth', 
                   'did not improve', 'within noise']
    neg_count = sum(content.lower().count(m.lower()) for m in neg_markers)
    print(f'\n负面结果标记数: {neg_count}')
    
    # 列出所有 H2 标题（每次实验）
    entries = [l for l in content.splitlines() if l.startswith('## ')]
    print(f'\n实验条目 ({len(entries)}):')
    for e in entries[:15]:
        print(' ', e)

**这份文档的模板（四段式）是可抄的**：
- Motivation：为什么试
- What changed：diff 粒度
- Results：含负面结果
- Conclusion：keep / discard / 部分 keep

详见根目录 [`skills/iterative-experiment.md`](../skills/iterative-experiment.md)——已把这份体例提炼成通用 skill。

## 5. 热点文件 — `base_train.py` 是 leaderboard 的落脚点

In [ ]:
names = [n for n in git('log', '--name-only', '--pretty=format:').splitlines() if n]
for f, c in Counter(names).most_common(12):
    bar = '█' * min(c, 60)
    print(f'  {c:>3}  {f:<36}  {bar}')

**排序**：
- `scripts/base_train.py` (56) = 每次 leaderboard 记录都在这
- `README.md` (45) = 每次 leaderboard 更新 README 表格
- `nanochat/gpt.py` (36) = 模型定义，fp8/smear/backout 都落在这
- `dev/LOG.md` (36) = 实验日志
- `scripts/chat_sft.py` (24) = SFT 入口

## 6. 贡献者结构 — Karpathy-centric 但有第二维护者

In [ ]:
authors = git('log', '--pretty=format:%an').splitlines()
total = len(authors)
for a, n in Counter(authors).most_common(10):
    pct = n * 100 / total
    bar = '█' * int(pct / 2)
    print(f'  {a:<30} {n:>4} ({pct:5.1f}%)  {bar}')

**svlandeg** 是真正的第二维护者（40 commits，覆盖 CI / CPU 兼容 / typo / bug fix）——这种"一位可信长期合作者"的结构比"100 个偶尔贡献者"更健康。

## 7. Dependency 洁癖 — 2026-03-26 的 `a445144`

In [ ]:
print(git('show', '-s', '--format=%h %ad%n%s%n%n%b', 'a445144', '--date=short'))

**学到的**：依赖精简的理由**不是磁盘空间，是供应链攻击面**。成熟开源项目都该在某一阶段做一次类似清理。

## 8. Takeaways

1. **数据 > 算法**：一次 ClimbMix 换血 = 所有其他加速累加。任何训练项目，先看数据。
2. **leaderboard + dev/LOG.md** 是长期开源项目的双驱动：leaderboard 拉流量，LOG.md 拉可信度。
3. **commit msg 里写情绪**（`i suffered` / `dependencies bad bad bad`）是可读 git history 的核心实践。
4. **单节点 + 单旋钮** 是长期护城河——抵制扩张诱惑。
5. **autoresearch → nanochat 闭环** 证明 agent-driven 改进可以 merge 进主线，不止是 demo。

完整分析见 [`nanochat.html`](nanochat.html)。相关 skill：[`skills/iterative-experiment.md`](../skills/iterative-experiment.md)。